# Working with atmoz.surface.AirNow

**Author:** Maurice Roots  
**Contact:** mrmoroots@gmail.com  

---

## Summary

This notebook demonstrates how to acquire, explore, and analyze surface ozone data from the U.S. Environmental Protection Agency (EPA) using the `atmoz.surface` module.

## What is EPA AQS?

The **Air Quality System (AQS)** is the EPA's repository of ambient air quality data collected by federal, state, local, and tribal monitoring agencies across thousands of sites. Data are publicly available at no cost and cover criteria pollutants including ozone, PM2.5, PM10, NO₂, SO₂, and CO.

**AirNow** is a companion real-time system that publishes near-real-time (hourly) air quality data. While AQS is the authoritative long-term archive, AirNow is better suited for monitoring current conditions.

## What is Surface Ozone?

Ground-level ozone (O₃) is a **secondary pollutant** — it is not emitted directly but forms when nitrogen oxides (NOₓ) and volatile organic compounds (VOCs) react in sunlight. Unlike stratospheric ozone (the "ozone layer"), ground-level ozone is harmful to human health and vegetation.

**The EPA National Ambient Air Quality Standard (NAAQS) for ozone is 70 ppb**, based on the annual fourth-highest daily maximum 8-hour average concentration over three years. Any area exceeding this threshold is designated as non-attainment and faces regulatory action.

## Table of Contents

1. [Data Acquisition](#section1)
2. [Data Structure & Preprocessing](#section2)
3. [Mapping EPA Monitoring Sites](#section3)
4. [Geographic Slicing: Mid-Atlantic Region](#section4)
5. [Data Analysis](#section5)
6. [Conclusion](#section6)

## Prerequisites

```
pip install git+https://github.com/moroots/atmoz
```
Dependencies: `geopandas`, `plotly`, `matplotlib`

In [ ]:
from atmoz.surface import AirNow
from atmoz.surface import utilities as su

import pickle
from pathlib import Path

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

## 1. Data Acquisition <a class="anchor" id="section1"></a>

### 1a. AQS Pre-Generated Files

The EPA publishes **pre-generated annual CSV files** for each criteria pollutant through its AQS Data Mart. These files are available without an API key and are ideal for historical analysis.

`AirNow.download(endpoint="aqs", ...)` fetches these files automatically:

| Parameter | Description |
|-----------|-------------|
| `parameters` | Pollutant name, e.g. `["Ozone"]` |
| `resolutions` | `"hourly"` or `"daily"` |
| `years` | Calendar year(s) to download, e.g. `"2025"` |

The downloaded file for hourly ozone is named `hourly_44201_{year}.csv` (44201 is the EPA parameter code for ozone).

In [ ]:
filename = "AirNow_download.pkl"
if Path(filename).is_file():
    with open(filename, "rb") as f:
        surface = pickle.load(f)
else:
    surface = AirNow.download(endpoint="aqs", parameters=["Ozone"], resolutions="hourly", years="2025")
    with open(filename, "wb") as f:
        pickle.dump(surface, f)

### 1b. Live AirNow API (Bonus)

The `AirNow` class can also query the **live AirNow API** for near-real-time data covering the past 24–72 hours. A free API key is available at [https://docs.airnowapi.org/](https://docs.airnowapi.org/).

The `BBOX` parameter defines the geographic bounding box as `[west, south, east, north]`.

In [ ]:
# Uncomment and replace "YOUR_KEY" to use the live AirNow API
#
# result = AirNow.download(
#     endpoint="airnow",
#     api_key="YOUR_KEY",
#     parameters=["OZONE"],
#     BBOX=["-80.0", "36.5", "-73.0", "42.5"],   # [west, south, east, north]
#     start_date="2025-08-01",
#     end_date="2025-08-07",
# )
# result.data      # dict keyed by parameter name → DataFrame
# result.metadata  # DataFrame with one row per monitoring station

## 2. Data Structure & Preprocessing <a class="anchor" id="section2"></a>

`surface` is a Python dictionary where each key is a CSV filename and the value is a `pandas.DataFrame`. The cell below extracts the DataFrame and creates a combined `Datetime` column from the separate date and time columns.

In [ ]:
# Extract the ozone DataFrame from the surface dict
df = surface["hourly_44201_2025.csv"].copy()

list(df.columns)

### Key Columns

| Column | Description |
|--------|-------------|
| `State Code` / `County Code` / `Site Num` | Three-part unique station identifier |
| `Sample Measurement` | Ozone concentration in **ppb** |
| `Date GMT` / `Time GMT` | Observation timestamp in UTC |
| `Latitude` / `Longitude` | Station geographic coordinates |
| `POC` | Parameter Occurrence Code — distinguishes multiple instruments at the same site |
| `Datum` | Coordinate reference datum (typically NAD83 or WGS84) |

In [ ]:
# Combine the separate Date GMT and Time GMT columns into a single Datetime column
df["Datetime"] = su.make_datetime(df)

df[["Date GMT", "Time GMT", "Datetime", "Sample Measurement", "State Name", "County Name"]].head()

Before plotting, we extract **station metadata** (one row per unique site) and convert it to a `GeoDataFrame` so we can perform geographic operations like bounding-box slicing.

In [ ]:
meta     = su.extract_metadata(df)
gdf_meta = su.to_gdf(meta)

# Sanity check: each station should have a unique lat/lon pair
su.count_duplicates(gdf_meta, cols=["Latitude", "Longitude"])

## 3. Mapping EPA Monitoring Sites <a class="anchor" id="section3"></a>

The interactive map below shows every site reporting hourly ozone data in 2025. The U.S. has a dense monitoring network — hover over a point to see the site's state, county, and coordinates.

In [ ]:
su.site_map(
    instruments={"Ozone Sites": gdf_meta},
    title="EPA Air Quality Monitoring Sites — Ozone (2025)",
    map={"zoom": 3, "pitch": 60},
    hover_cols=["Site Num", "State Name", "County Name", "Latitude", "Longitude"],
)

## 4. Geographic Slicing: Mid-Atlantic Region <a class="anchor" id="section4"></a>

Working with the full national dataset is often unnecessary. `su.slice_bbox()` clips the station GeoDataFrame to a bounding box, and `su.df_geo_slice()` filters the full hourly DataFrame to only those stations.

**Why Mid-Atlantic?** This region — spanning Maryland, Virginia, Delaware, Pennsylvania, New Jersey, and New York — is densely populated and regularly experiences elevated summer ozone from traffic emissions, industrial sources, and high temperatures.

In [ ]:
# Bounding box: lon [west, east], lat [south, north]
bbox = {"lon": [-80.0, -73.0], "lat": [36.5, 42.5]}

midAtlantic = su.slice_bbox(gdf_meta, bbox=bbox)

In [ ]:
df_midAtlantic = su.df_geo_slice(df, midAtlantic)

In [ ]:
midAtlantic

In [ ]:
su.site_map(
    instruments={"Mid-Atlantic": midAtlantic},
    title="Mid-Atlantic EPA Air Quality Sites",
    map={"zoom": 6, "pitch": 50},
    bbox={"lon": [-80.0, -73.0], "lat": [36.5, 42.5]},
    hover_cols=[
        "State Code", "County Code", "Site Num",
        "State Name", "County Name",
        "Latitude", "Longitude", "Parameter Name",
    ],
)

## 5. Data Analysis <a class="anchor" id="section5"></a>

### 5a. Split by Parameter

`su.split_by_parameter()` groups the data by pollutant name. This download requested **Ozone** only, so there is one key. Downloading multiple parameters (e.g. `["Ozone", "PM25"]`) would produce multiple keys.

In [ ]:
dict_midAtlantic = su.split_by_parameter(df_midAtlantic)

In [ ]:
dict_midAtlantic.keys()

### 5b. Split by Station

`su.split_by_station()` groups the ozone DataFrame by the three-part station identifier: **(State Code, County Code, Site Num)**. This gives one DataFrame per monitoring site.

In [ ]:
dict_midAtlantic_by_station = su.split_by_station(dict_midAtlantic["Ozone"])

print(f"{len(dict_midAtlantic_by_station)} unique stations in the Mid-Atlantic ozone dataset")
print("\nFirst five station keys:")
for k in list(dict_midAtlantic_by_station.keys())[:5]:
    g = dict_midAtlantic_by_station[k]
    state, county, site = k
    print(f"  ({state}, {county}, {site})  →  {g['County Name'].iloc[0]}, "
          f"{g['State Name'].iloc[0]}  [{len(g)} hourly obs]")

### 5c. Single-Station Analysis

The cells below select one station and build four visualizations. Change `station_key` to any tuple from the list above to explore a different site.

In [ ]:
station_key = sorted(dict_midAtlantic_by_station.keys())[0]
df_station  = dict_midAtlantic_by_station[station_key].sort_values("Datetime").copy()

state, county, site = station_key
county_name = df_station["County Name"].iloc[0]
state_name  = df_station["State Name"].iloc[0]
print(f"Station: {county_name}, {state_name}  (State {state} · County {county} · Site {site})")
print(f"Observations: {len(df_station)}  |  "
      f"{df_station['Datetime'].min().date()} → {df_station['Datetime'].max().date()}")

### 5d. Ozone Time Series

Raw hourly ozone (light) with an 8-hour rolling mean (bold) and the EPA 70 ppb threshold.

> **Why 8 hours?** The NAAQS ozone standard is based on the *annual fourth-highest daily maximum 8-hour average* over three years. A rolling 8-hour window captures the sustained exposure a person experiences outdoors during peak ozone hours.

In [ ]:
# 8-hour rolling mean on a datetime-indexed series (min 6 of 8 obs required)
ts = df_station.set_index("Datetime")["Sample Measurement"]
rolling_8h = ts.rolling("8h", min_periods=6).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ts.index, ts.values, color="#7195AB", alpha=0.45, linewidth=0.8, label="Hourly Ozone")
ax.plot(rolling_8h.index, rolling_8h.values, color="#CD6091", linewidth=2.2, label="8-hr Rolling Mean")
ax.axhline(70, color="#E76F51", linestyle="--", linewidth=1.6, label="EPA NAAQS O₃ (70 ppb)")
ax.set_xlabel("Date (UTC)", fontsize=12)
ax.set_ylabel("Ozone (ppb)", fontsize=12)
ax.set_title(f"Surface Ozone — {county_name}, {state_name}", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, linestyle="--", alpha=0.5)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### 5e. Diurnal Pattern

Average ozone by hour of day with ±1 standard deviation. Ozone is a **photochemical** pollutant — it typically reaches a minimum before sunrise and peaks in the mid-afternoon when solar radiation is highest.

In [ ]:
df_station["Hour"] = df_station["Datetime"].dt.hour
diurnal = (
    df_station.groupby("Hour")["Sample Measurement"]
    .agg(["mean", "std"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(diurnal["Hour"], diurnal["mean"], color="#CD6091", linewidth=2.2, marker="o", label="Mean O₃")
ax.fill_between(
    diurnal["Hour"],
    diurnal["mean"] - diurnal["std"],
    diurnal["mean"] + diurnal["std"],
    alpha=0.2, color="#CD6091", label="±1 std dev",
)
ax.axhline(70, color="#E76F51", linestyle="--", linewidth=1.5, label="EPA NAAQS (70 ppb)")
ax.set_xlabel("Hour of Day (UTC)", fontsize=12)
ax.set_ylabel("Ozone (ppb)", fontsize=12)
ax.set_title(f"Diurnal Ozone Pattern — {county_name}, {state_name}", fontsize=14, fontweight="bold")
ax.set_xticks(range(0, 24, 2))
ax.legend(fontsize=11)
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### 5f. NAAQS Exceedance Analysis

How many days had a peak 8-hour ozone average above 70 ppb? The **daily maximum 8-hour mean** is the metric used for EPA attainment/non-attainment designations.

In [ ]:
ts_indexed = df_station.set_index("Datetime").sort_index()["Sample Measurement"]
rolling     = ts_indexed.rolling("8h", min_periods=6).mean()
daily_max   = rolling.resample("1D").max().dropna()

exceedances = (daily_max > 70).sum()
total_days  = len(daily_max)
pct         = 100 * exceedances / total_days if total_days else 0.0

print(f"Days with peak 8-hr mean O₃ > 70 ppb: {exceedances} of {total_days} ({pct:.1f}%)")

if exceedances > 0:
    print("\nExceedance dates and peak 8-hr values:")
    print(
        daily_max[daily_max > 70]
        .round(1)
        .rename("Peak 8-hr O₃ (ppb)")
        .to_frame()
        .to_string()
    )

### 5g. Summary Statistics

A quick statistical profile of the hourly ozone record.

- **Mean > Median** → the distribution is right-skewed, pulled up by high-ozone episodes.  
- **Skewness > 1** → strong right tail; ozone spikes are infrequent but large.  
- **% Hours > 70 ppb** → how often did hourly readings exceed the NAAQS threshold?

In [ ]:
conc = df_station["Sample Measurement"]
summary = pd.Series({
    "Count (hours)":    int(conc.count()),
    "Mean (ppb)":       conc.mean(),
    "Median (ppb)":     conc.median(),
    "Std Dev (ppb)":    conc.std(),
    "Min (ppb)":        conc.min(),
    "Max (ppb)":        conc.max(),
    "Skewness":         conc.skew(),
    "% Hours > 70 ppb": 100 * (conc > 70).mean(),
}).round(2)

summary

## 6. Conclusion <a class="anchor" id="section6"></a>

In this tutorial we used `atmoz.surface` to work with EPA ozone data end-to-end:

- **Downloaded** annual AQS pre-generated ozone CSV files with `AirNow.download(endpoint="aqs")`  
- **Mapped** thousands of monitoring stations interactively using `su.site_map()`  
- **Sliced** the dataset to the Mid-Atlantic region with `su.slice_bbox()` and `su.df_geo_slice()`  
- **Analyzed** ozone at an individual station: time series with EPA threshold, diurnal cycle, exceedance events, and summary statistics

### Suggested Next Steps

| Idea | How |
|------|-----|
| Analyze a different pollutant | Change `parameters=["PM25"]` in `AirNow.download()` |
| Compare multiple stations | Loop over `dict_midAtlantic_by_station` instead of selecting one |
| Look at wildfire smoke days | Filter for days when the diurnal peak shifts unusually early |
| Compare with model output | See the **GEOS-CF tutorial** (`atmoz_tutorial_geoscf.ipynb`) |
| Live real-time data | Use `AirNow.download(endpoint="airnow")` with a free API key from airnow.gov |